<a href="https://colab.research.google.com/github/omarrgohary/Phishing-Emails-URLs-Detection/blob/main/Image_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08.csv to CEAS_08.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving PhiUSIIL_Phishing_URL_Dataset.csv to PhiUSIIL_Phishing_URL_Dataset.csv


In [ ]:
!pip install torchtext transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.8 MB/s eta 0:00:00


In [ ]:
import csv
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm.auto import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
email_df = pd.read_csv(
    "/content/CEAS_08.csv",
    engine="python",
    on_bad_lines="skip"
)

url_df = pd.read_csv(
    "/content/PhiUSIIL_Phishing_URL_Dataset.csv",
    engine="python"
)

print("Email shape:", email_df.shape)
print("URL shape:", url_df.shape)


Email shape: (39153, 7)
URL shape: (235795, 56)


In [ ]:
LEAKAGE_PATTERNS = [
    r"\bspam\b",
    r"\bphish\b",
    r"\bphishing\b",
    r"\bham\b"
]

def remove_leakage(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    for p in LEAKAGE_PATTERNS:
        text = re.sub(p, "", text)
    return text

email_df["subject"] = email_df["subject"].apply(remove_leakage)
email_df["body"] = email_df["body"].apply(remove_leakage)
email_df["sender"] = email_df["sender"].astype(str)


In [ ]:
email_df["text_raw"] = (
    email_df["subject"].fillna("") + " " +
    email_df["body"].fillna("")
)

before = len(email_df)
email_df = email_df.drop_duplicates(subset="text_raw")
after = len(email_df)

print(f"Removed {before - after} duplicate emails")

email_df["text"] = (
    email_df["subject"].fillna("") + " [SEP] " +
    email_df["body"].fillna("")
)

email_texts = email_df["text"].tolist()
email_labels = email_df["label"].astype(int).tolist()


Removed 0 duplicate emails


In [ ]:
X_train_e, X_val_e, y_train_e, y_val_e = train_test_split(
    email_texts,
    email_labels,
    test_size=0.2,
    random_state=42,
    stratify=email_labels
)


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_emails(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_enc_e = tokenize_emails(X_train_e)
val_enc_e   = tokenize_emails(X_val_e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [ ]:
class EmailDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
class BERTEmailClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        cls = self.dropout(cls)
        return self.fc(cls)


In [ ]:
train_ds_e = EmailDataset(train_enc_e, y_train_e)
val_ds_e   = EmailDataset(val_enc_e, y_val_e)

train_loader_e = DataLoader(train_ds_e, batch_size=16, shuffle=True)
val_loader_e   = DataLoader(val_ds_e, batch_size=16)

email_model = BERTEmailClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(email_model.parameters(), lr=2e-5)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
def train_email_epoch(model, loader, epoch, total_epochs):
    model.train()
    total_loss = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Training]", leave=False)
    for batch in pbar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    return total_loss / len(loader)

def eval_email(model, loader, epoch, total_epochs):
    model.eval()
    preds, labels = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Evaluating]", leave=False)
    with torch.no_grad():
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["labels"].to(device)

            out = model(input_ids, mask)
            preds.extend(torch.argmax(out, 1).cpu().numpy())
            labels.extend(y.cpu().numpy())

    return {
        "acc": accuracy_score(labels, preds),
        "prec": precision_score(labels, preds),
        "rec": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


In [ ]:
EPOCHS = 3

for epoch in range(1, EPOCHS + 1):
    loss = train_email_epoch(email_model, train_loader_e, epoch, EPOCHS)
    metrics = eval_email(email_model, val_loader_e, epoch, EPOCHS)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Loss: {loss:.4f} | "
        f"Acc: {metrics['acc']:.4f} | "
        f"Prec: {metrics['prec']:.4f} | "
        f"Rec: {metrics['rec']:.4f} | "
        f"F1: {metrics['f1']:.4f}"
    )


Epoch 1/3 [Training]:   0%|          | 0/1958 [00:00<?, ?it/s]

Epoch 1/3 [Evaluating]:   0%|          | 0/490 [00:00<?, ?it/s]

Epoch 1/3 | Loss: 0.0254 | Acc: 0.9967 | Prec: 0.9948 | Rec: 0.9993 | F1: 0.9970


Epoch 2/3 [Training]:   0%|          | 0/1958 [00:00<?, ?it/s]

Epoch 2/3 [Evaluating]:   0%|          | 0/490 [00:00<?, ?it/s]

Epoch 2/3 | Loss: 0.0043 | Acc: 0.9980 | Prec: 0.9982 | Rec: 0.9982 | F1: 0.9982


Epoch 3/3 [Training]:   0%|          | 0/1958 [00:00<?, ?it/s]

Epoch 3/3 [Evaluating]:   0%|          | 0/490 [00:00<?, ?it/s]

Epoch 3/3 | Loss: 0.0024 | Acc: 0.9959 | Prec: 0.9934 | Rec: 0.9993 | F1: 0.9963


In [ ]:
enc = tokenizer(
    test_texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

email_model.eval()
with torch.no_grad():
    logits = email_model(
        enc["input_ids"].to(device),
        enc["attention_mask"].to(device)
    )

preds = torch.argmax(logits, dim=1).cpu().numpy()

print("\nEMAIL MODEL TEST RESULTS — FROM CSV\n")

for i, (_, row) in enumerate(test_email_samples.iterrows()):
    print("Subject:", row["subject"])
    print("Prediction:", "Phishing" if preds[i] == 1 else "Legitimate")
    print("True Label:", "Phishing" if row["label"] == 1 else "Legitimate")
    print("-" * 60)



EMAIL MODEL TEST RESULTS — FROM CSV

Subject: [sm-devel] subfolders untranslated
Prediction: Legitimate
True Label: Legitimate
------------------------------------------------------------
Subject: [uai] cfp: international symposium on practical cognitive agents and robots
Prediction: Legitimate
True Label: Legitimate
------------------------------------------------------------
Subject: a simple php interface to read photos from directories
Prediction: Legitimate
True Label: Legitimate
------------------------------------------------------------
Subject: the bigger tool
Prediction: Phishing
True Label: Phishing
------------------------------------------------------------
Subject: cnn.com daily top 10
Prediction: Phishing
True Label: Phishing
------------------------------------------------------------
Subject: satisfy your woman?s craving easily
Prediction: Phishing
True Label: Phishing
------------------------------------------------------------


In [ ]:
external_emails = [
    {
        "subject": "Urgent: Verify your bank account immediately",
        "body": "Your account has been suspended. Click the link below to verify your identity."
    },
    {
        "subject": "Meeting reminder",
        "body": "Just a reminder about our project meeting tomorrow at 10 AM."
    },
    {
        "subject": "Security alert from PayPal",
        "body": "We detected unusual activity. Login now to secure your account."
    }
]

processed_texts = []
for email in external_emails:
    subj = remove_leakage(email["subject"])
    body = remove_leakage(email["body"])
    processed_texts.append(subj + " [SEP] " + body)

enc = tokenizer(
    processed_texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

email_model.eval()
with torch.no_grad():
    logits = email_model(
        enc["input_ids"].to(device),
        enc["attention_mask"].to(device)
    )

preds = torch.argmax(logits, dim=1).cpu().numpy()

print("\nEMAIL MODEL — REAL-WORLD TEST RESULTS\n")

for i, email in enumerate(external_emails):
    print(f"Email {i+1}")
    print("Subject:", email["subject"])
    print("Prediction:", " Phishing" if preds[i] == 1 else " Legitimate")
    print("-" * 60)



EMAIL MODEL — REAL-WORLD TEST RESULTS

Email 1
Subject: Urgent: Verify your bank account immediately
Prediction:  Phishing
------------------------------------------------------------
Email 2
Subject: Meeting reminder
Prediction:  Legitimate
------------------------------------------------------------
Email 3
Subject: Security alert from PayPal
Prediction:  Phishing
------------------------------------------------------------


In [ ]:
url_texts = url_df["URL"].astype(str).tolist()

engineered_cols = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "ObfuscationRatio",
    "NoOfLettersInURL",
    "LetterRatioInURL",
    "NoOfDegitsInURL",
    "DegitRatioInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL",
    "IsHTTPS"
]

X_feats = url_df[engineered_cols].values
y_urls = url_df["label"].astype(int).values


In [ ]:
scaler = StandardScaler()
X_feats = scaler.fit_transform(X_feats)

X_url_tr, X_url_val, F_tr, F_val, y_tr, y_val = train_test_split(
    url_texts,
    X_feats,
    y_urls,
    test_size=0.2,
    random_state=42,
    stratify=y_urls
)


In [ ]:
all_chars = set()

for url in X_url_tr:
    for c in url:
        all_chars.add(c)

char2idx = {c: i + 1 for i, c in enumerate(sorted(all_chars))}
char2idx["<pad>"] = 0

idx2char = {i: c for c, i in char2idx.items()}

VOCAB_SIZE = len(char2idx)
print("Vocab size:", VOCAB_SIZE)


Vocab size: 60


In [ ]:
MAX_URL_LEN = 200

def encode_urls(urls, max_len=MAX_URL_LEN):
    encoded = []
    for url in urls:
        ids = [char2idx.get(c, 0) for c in url[:max_len]]
        if len(ids) < max_len:
            ids += [0] * (max_len - len(ids))
        encoded.append(ids)
    return torch.tensor(encoded, dtype=torch.long)


In [ ]:
class URLDataset(Dataset):
    def __init__(self, urls, features, labels):
        self.url_ids = encode_urls(urls)
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "url_ids": self.url_ids[idx],
            "features": self.features[idx],
            "labels": self.labels[idx]
        }


In [ ]:
class HybridURLModel(nn.Module):
    def __init__(self, vocab_size, feature_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.position = nn.Embedding(MAX_URL_LEN, 64)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 32),
            nn.ReLU()
        )

        self.classifier = nn.Linear(64 + 32, 2)

    def forward(self, url_ids, features):
        batch_size, seq_len = url_ids.shape

        pos_ids = torch.arange(seq_len, device=url_ids.device)\
                         .unsqueeze(0).repeat(batch_size, 1)

        x = self.embedding(url_ids) + self.position(pos_ids)
        x = self.encoder(x)
        x = x.mean(dim=1)

        f = self.feature_mlp(features)
        x = torch.cat([x, f], dim=1)

        return self.classifier(x)


In [ ]:
train_ds_u = URLDataset(X_url_tr, F_tr, y_tr)
val_ds_u   = URLDataset(X_url_val, F_val, y_val)

train_loader_u = DataLoader(train_ds_u, batch_size=32, shuffle=True)
val_loader_u   = DataLoader(val_ds_u, batch_size=32)

url_model = HybridURLModel(VOCAB_SIZE, F_tr.shape[1]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(url_model.parameters(), lr=1e-3)


In [ ]:
def train_url_epoch(epoch, total_epochs):
    url_model.train()
    pbar = tqdm(train_loader_u, desc=f"Epoch {epoch}/{total_epochs} [URL Training]", leave=False)

    for batch in pbar:
        optimizer.zero_grad()

        outputs = url_model(
            batch["url_ids"].to(device),
            batch["features"].to(device)
        )

        loss = criterion(outputs, batch["labels"].to(device))
        loss.backward()
        optimizer.step()

        pbar.set_postfix(loss=loss.item())

def eval_url(epoch, total_epochs):
    url_model.eval()
    preds, labels = [], []

    pbar = tqdm(val_loader_u, desc=f"Epoch {epoch}/{total_epochs} [URL Eval]", leave=False)
    with torch.no_grad():
        for batch in pbar:
            outputs = url_model(
                batch["url_ids"].to(device),
                batch["features"].to(device)
            )

            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    return {
        "acc": accuracy_score(labels, preds),
        "prec": precision_score(labels, preds),
        "rec": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


In [ ]:
URL_EPOCHS = 3

for epoch in range(1, URL_EPOCHS + 1):
    train_url_epoch(epoch, URL_EPOCHS)
    metrics = eval_url(epoch, URL_EPOCHS)

    print(
        f"Epoch {epoch}/{URL_EPOCHS} | "
        f"Acc: {metrics['acc']:.4f} | "
        f"Prec: {metrics['prec']:.4f} | "
        f"Rec: {metrics['rec']:.4f} | "
        f"F1: {metrics['f1']:.4f}"
    )


Epoch 1/3 [URL Training]:   0%|          | 0/5895 [00:00<?, ?it/s]

Epoch 1/3 [URL Eval]:   0%|          | 0/1474 [00:00<?, ?it/s]

Epoch 1/3 | Acc: 0.9972 | Prec: 0.9953 | Rec: 0.9999 | F1: 0.9976


Epoch 2/3 [URL Training]:   0%|          | 0/5895 [00:00<?, ?it/s]

Epoch 2/3 [URL Eval]:   0%|          | 0/1474 [00:00<?, ?it/s]

Epoch 2/3 | Acc: 0.9972 | Prec: 0.9953 | Rec: 0.9999 | F1: 0.9976


Epoch 3/3 [URL Training]:   0%|          | 0/5895 [00:00<?, ?it/s]

Epoch 3/3 [URL Eval]:   0%|          | 0/1474 [00:00<?, ?it/s]

Epoch 3/3 | Acc: 0.9975 | Prec: 0.9958 | Rec: 0.9999 | F1: 0.9978


In [ ]:
test_samples = pd.concat([
    url_df[url_df["label"] == 0].sample(3, random_state=42),
    url_df[url_df["label"] == 1].sample(3, random_state=42)
])

test_urls = test_samples["URL"].astype(str).tolist()

test_features = test_samples[engineered_cols].values
test_features = scaler.transform(test_features)
test_features = torch.tensor(test_features, dtype=torch.float32)

test_url_ids = encode_urls(test_urls)

url_model.eval()
with torch.no_grad():
    logits = url_model(
        test_url_ids.to(device),
        test_features.to(device)
    )

preds = torch.argmax(logits, dim=1).cpu().numpy()

print("\nURL MODEL TEST RESULTS — FROM CSV\n")

for i, (_, row) in enumerate(test_samples.iterrows()):
    print(f"Sample {i+1}")
    print("URL:", row["URL"])
    print("Prediction:", "Phishing" if preds[i] == 1 else "Legitimate")
    print("True Label:", "Phishing" if row["label"] == 1 else "Legitimate")
    print("-" * 60)



URL MODEL TEST RESULTS — FROM CSV

Sample 1
URL: http://www.soeme.com
Prediction: Legitimate
True Label: Legitimate
------------------------------------------------------------
Sample 2
URL: http://www.v2cde1b0d66c767a23cfb34c14552836d3.ws
Prediction: Legitimate
True Label: Legitimate
------------------------------------------------------------
Sample 3
URL: https://locateme.co.nz/wp-content/jam/ichiemagiksouthwest123.html
Prediction: Legitimate
True Label: Legitimate
------------------------------------------------------------
Sample 4
URL: https://www.atelierozmoz.be
Prediction: Phishing
True Label: Phishing
------------------------------------------------------------
Sample 5
URL: https://www.diemon.com
Prediction: Phishing
True Label: Phishing
------------------------------------------------------------
Sample 6
URL: https://www.wausauschools.org
Prediction: Phishing
True Label: Phishing
------------------------------------------------------------
